In [3]:
from google.colab import files

uploaded = files.upload()

Saving gk_qna_dataset.csv to gk_qna_dataset (1).csv


In [4]:
import pandas as pd
import numpy as np
import re

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

In [5]:
df = pd.read_csv("gk_qna_dataset.csv")

df.head()

,question,answer
0,"What is the capital of ""France""?",Paris
1,"What is the capital of ""Germany""?",Berlin
2,"What is the capital of ""Italy""?",Rome
3,"What is the capital of ""Spain""?",Madrid
4,"What is the capital of ""Japan""?",Tokyo


In [6]:
print(df.shape)

df.info()

(171, 2)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  171 non-null    object
 1   answer    171 non-null    object
dtypes: object(2)
memory usage: 2.8+ KB


In [7]:
df.isnull().sum()

,0
question,0
answer,0


In [8]:
df["question"][0]

'What is the capital of "France"?'

In [9]:
df["answer"][0]

'Paris'

In [10]:
def tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return text.split()

In [11]:
tokenizer(df["question"][0])

['what', 'is', 'the', 'capital', 'of', 'france']

In [12]:
vocab = {
    "<PAD>":0,
    "<UNK>":1
}

In [13]:
def build_vocab(row):

    question = tokenizer(row["question"])

    for word in question:

        if word not in vocab:
            vocab[word] = len(vocab)

In [14]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
166,None
167,None
168,None
169,None


In [15]:
print(vocab)

{'<PAD>': 0, '<UNK>': 1, 'what': 2, 'is': 3, 'the': 4, 'capital': 5, 'of': 6, 'france': 7, 'germany': 8, 'italy': 9, 'spain': 10, 'japan': 11, 'canada': 12, 'brazil': 13, 'australia': 14, 'india': 15, 'china': 16, 'russia': 17, 'united': 18, 'states': 19, 'mexico': 20, 'egypt': 21, 'turkey': 22, 'argentina': 23, 'south': 24, 'korea': 25, 'indonesia': 26, 'pakistan': 27, 'bangladesh': 28, 'nepal': 29, 'sri': 30, 'lanka': 31, 'thailand': 32, 'malaysia': 33, 'vietnam': 34, 'uae': 35, 'iran': 36, 'iraq': 37, 'saudi': 38, 'arabia': 39, 'nigeria': 40, 'kenya': 41, 'chile': 42, 'who': 43, 'discovered': 44, 'gravity': 45, 'invented': 46, 'telephone': 47, 'h2o': 48, 'commonly': 49, 'called': 50, 'which': 51, 'gas': 52, 'do': 53, 'plants': 54, 'absorb': 55, 'planet': 56, 'known': 57, 'as': 58, 'red': 59, 'painted': 60, 'mona': 61, 'lisa': 62, 'largest': 63, 'ocean': 64, 'developed': 65, 'theory': 66, 'relativity': 67, 'ai': 68, 'short': 69, 'for': 70, 'ml': 71, 'nlp': 72, 'does': 73, 'cpu': 74, 

In [16]:
print("Vocabulary Size =",len(vocab))

Vocabulary Size = 141


In [17]:
answer_vocab={}

In [18]:
for ans in df["answer"].unique():

    answer_vocab[ans]=len(answer_vocab)

In [19]:
answer_vocab

{'Paris': 0,
 'Berlin': 1,
 'Rome': 2,
 'Madrid': 3,
 'Tokyo': 4,
 'Ottawa': 5,
 'Brasilia': 6,
 'Canberra': 7,
 'New Delhi': 8,
 'Beijing': 9,
 'Moscow': 10,
 'Washington DC': 11,
 'Mexico City': 12,
 'Cairo': 13,
 'Ankara': 14,
 'Buenos Aires': 15,
 'Seoul': 16,
 'Jakarta': 17,
 'Islamabad': 18,
 'Dhaka': 19,
 'Kathmandu': 20,
 'Colombo': 21,
 'Bangkok': 22,
 'Kuala Lumpur': 23,
 'Hanoi': 24,
 'Abu Dhabi': 25,
 'Tehran': 26,
 'Baghdad': 27,
 'Riyadh': 28,
 'Abuja': 29,
 'Nairobi': 30,
 'Santiago': 31,
 'Isaac Newton': 32,
 'Alexander Graham Bell': 33,
 'Water': 34,
 'Carbon dioxide': 35,
 'Mars': 36,
 'Leonardo da Vinci': 37,
 'Pacific Ocean': 38,
 'Albert Einstein': 39,
 'Artificial Intelligence': 40,
 'Machine Learning': 41,
 'Natural Language Processing': 42,
 'Central Processing Unit': 43,
 'Computer memory': 44,
 'Graphics Processing Unit': 45,
 'Web address': 46,
 'HyperText Transfer Protocol': 47,
 'Data exchange': 48,
 'Database query language': 49,
 'Application Programming 

In [20]:
answer_vocab

{'Paris': 0,
 'Berlin': 1,
 'Rome': 2,
 'Madrid': 3,
 'Tokyo': 4,
 'Ottawa': 5,
 'Brasilia': 6,
 'Canberra': 7,
 'New Delhi': 8,
 'Beijing': 9,
 'Moscow': 10,
 'Washington DC': 11,
 'Mexico City': 12,
 'Cairo': 13,
 'Ankara': 14,
 'Buenos Aires': 15,
 'Seoul': 16,
 'Jakarta': 17,
 'Islamabad': 18,
 'Dhaka': 19,
 'Kathmandu': 20,
 'Colombo': 21,
 'Bangkok': 22,
 'Kuala Lumpur': 23,
 'Hanoi': 24,
 'Abu Dhabi': 25,
 'Tehran': 26,
 'Baghdad': 27,
 'Riyadh': 28,
 'Abuja': 29,
 'Nairobi': 30,
 'Santiago': 31,
 'Isaac Newton': 32,
 'Alexander Graham Bell': 33,
 'Water': 34,
 'Carbon dioxide': 35,
 'Mars': 36,
 'Leonardo da Vinci': 37,
 'Pacific Ocean': 38,
 'Albert Einstein': 39,
 'Artificial Intelligence': 40,
 'Machine Learning': 41,
 'Natural Language Processing': 42,
 'Central Processing Unit': 43,
 'Computer memory': 44,
 'Graphics Processing Unit': 45,
 'Web address': 46,
 'HyperText Transfer Protocol': 47,
 'Data exchange': 48,
 'Database query language': 49,
 'Application Programming 

In [21]:
idx2answer={}

for k,v in answer_vocab.items():

    idx2answer[v]=k

In [22]:
idx2answer
MAX_LEN=10

In [23]:
def text_to_indices(text):

    tokens=tokenizer(text)

    ids=[]

    for word in tokens:

        ids.append(vocab.get(word,1))

    if len(ids)<MAX_LEN:

        ids=ids+[0]*(MAX_LEN-len(ids))

    else:

        ids=ids[:MAX_LEN]

    return ids

In [24]:
text_to_indices(df["question"][0])

[2, 3, 4, 5, 6, 7, 0, 0, 0, 0]

In [25]:
class GKDataset(Dataset):

    def __init__(self,df):

        self.df=df

    def __len__(self):

        return len(self.df)

    def __getitem__(self,idx):

        question=self.df.iloc[idx]["question"]
        answer=self.df.iloc[idx]["answer"]

        x=torch.tensor(text_to_indices(question))

        y=torch.tensor(answer_vocab[answer])

        return x,y

In [28]:
dataset=GKDataset(df)
dataset[0]
loader=DataLoader(dataset,batch_size=8,shuffle=True)

In [29]:
class SimpleRNN(nn.Module):

    def __init__(self,vocab_size,embed_size=64,hidden_size=128,num_classes=len(answer_vocab)):

        super().__init__()

        self.embedding=nn.Embedding(vocab_size,embed_size,padding_idx=0)

        self.rnn=nn.RNN(embed_size,
                        hidden_size,
                        batch_first=True)

        self.fc=nn.Linear(hidden_size,
                          num_classes)

    def forward(self,x):

        x=self.embedding(x)

        output,hidden=self.rnn(x)

        hidden=hidden.squeeze(0)

        out=self.fc(hidden)

        return out

In [30]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

device
model=SimpleRNN(len(vocab)).to(device)

In [31]:
criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [32]:
epochs=50

for epoch in range(epochs):

    model.train()

    total_loss=0

    for x,y in loader:

        x=x.to(device)

        y=y.to(device)

        optimizer.zero_grad()

        pred=model(x)

        loss=criterion(pred,y)

        loss.backward()

        optimizer.step()

        total_loss+=loss.item()

    print(f"Epoch {epoch+1}/{epochs} Loss:{total_loss:.4f}")

Epoch 1/50 Loss:93.5406
Epoch 2/50 Loss:86.0230
Epoch 3/50 Loss:75.2032
Epoch 4/50 Loss:65.3648
Epoch 5/50 Loss:58.2061
Epoch 6/50 Loss:51.1678
Epoch 7/50 Loss:45.3261
Epoch 8/50 Loss:38.4615
Epoch 9/50 Loss:34.4590
Epoch 10/50 Loss:30.1601
Epoch 11/50 Loss:25.9724
Epoch 12/50 Loss:23.2418
Epoch 13/50 Loss:20.3274
Epoch 14/50 Loss:17.5145
Epoch 15/50 Loss:14.8992
Epoch 16/50 Loss:12.6964
Epoch 17/50 Loss:10.9343
Epoch 18/50 Loss:9.5786
Epoch 19/50 Loss:8.5951
Epoch 20/50 Loss:7.4353
Epoch 21/50 Loss:6.4406
Epoch 22/50 Loss:5.6091
Epoch 23/50 Loss:4.8400
Epoch 24/50 Loss:4.3501
Epoch 25/50 Loss:3.7800
Epoch 26/50 Loss:3.5338
Epoch 27/50 Loss:3.1421
Epoch 28/50 Loss:2.7473
Epoch 29/50 Loss:2.3677
Epoch 30/50 Loss:2.0955
Epoch 31/50 Loss:1.9393
Epoch 32/50 Loss:1.7472
Epoch 33/50 Loss:1.5859
Epoch 34/50 Loss:1.4934
Epoch 35/50 Loss:1.3512
Epoch 36/50 Loss:1.2626
Epoch 37/50 Loss:1.1873
Epoch 38/50 Loss:1.1358
Epoch 39/50 Loss:1.0461
Epoch 40/50 Loss:0.9649
Epoch 41/50 Loss:0.9127
Epoch 42

In [33]:
def predict(question):

    model.eval()

    x=torch.tensor(text_to_indices(question)).unsqueeze(0).to(device)

    with torch.no_grad():

        pred=model(x)

        idx=torch.argmax(pred,1).item()

    return idx2answer[idx]

In [35]:
predict("What is the capital of France?")


'Paris'

In [36]:
predict("What is the capital of Japan?")

'Tokyo'

In [ ]:
while True:

    q=input("Ask Question : ")

    if q.lower()=="exit":
        break

    print("Answer :",predict(q))

Ask Question : what is the capital of japan
Answer : Tokyo
